**12장 – 텐서플로를 사용한 사용자 정의 모델과 훈련**

### 설정

먼저 필요한 모듈을 임포트 한다. 맷플롯립 그래프를 인라인으로 출력하도록 만들고 그림을 저장하는 함수를 준비 하자. 또한 파이썬 버전이 3.5 이상인지 확인 하자(파이썬 2.x에서도 동작하지만 곧 지원이 중단되므로 파이썬 3을 사용하는 것이 좋다). 사이킷런 버전이 0.20 이상인지와 텐서플로 버전이 2.0 이상인지 확인한다.

In [ ]:
# 파이썬 ≥3.5 필수
import sys
assert sys.version_info >= (3, 5)

# 사이킷런 ≥0.20 필수
import sklearn
assert sklearn.__version__ >= "0.20"

# 이 노트북은 텐서플로 ≥2.4이 필요합
# 2.x 버전은 대부분 동일한 결과를 만들지만 몇 가지 버그가 있다.
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.11"

# 공통 모듈 임포트
import numpy as np
import os

# 노트북 실행 결과를 동일하게 유지하기 위해
np.random.seed(42)
tf.random.set_seed(42)

# 깔끔한 그래프 출력을 위해
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# 그림을 저장할 위치
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "deep"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("그림 저장:", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)


## 텐서와 연산

### 텐서

In [ ]:
l = [[1,2,3],[4,5,6]]
np.array(l)

In [ ]:
t=tf.constant([[1., 2., 3.], [4., 5., 6.]]) # 행렬
t

In [ ]:
t.numpy()

**np.ndarray의 기본 데이터 타입은 64bit이지만, tensor의 기본 데이터 타입은 32bit 이다**

In [ ]:
tf.constant(42) # 스칼라

Tensor는 np.ndarray와 상당히 유사하지만 스칼라 값도 가질수 있다.

In [ ]:
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])
t

In [ ]:
t.shape

In [ ]:
t.dtype

### 인덱싱과 슬라이싱

In [ ]:
t[:, 1:]

In [ ]:
t[..., 2, tf.newaxis]

... 은 모든 요소, 즉 t[:, :, :]를 말하며, 가운데 숫자 2는 index로 이 지점의 요소를 선택한다.

In [ ]:
t[:, 1, tf.newaxis]

tf.newaxis 는 새로운 차원을 더하는 것으로, 위 단계에서 전체 요소중 인덱스가 지정하는 요소를 모아 1차원의 tensor를 2차원의 tensor로 바꾼다. 가령 아래처럼 tf.newaxis가 없는 경우와 비교 해 보면 알 수 있다.

In [ ]:
t[..., 1:, tf.newaxis]

### 연산

In [ ]:
t + 10

In [ ]:
tf.square(t)

In [ ]:
t

#### 설명 t @ tf.transpose(T)
행렬곱셈(matrix multiplication), tf.matmul()

In [ ]:
t

In [ ]:
tf.transpose(t)


In [ ]:
t@tf.transpose(t)

### 넘파이 변환

In [ ]:
a = np.array([2., 4., 5.])
tf.constant(a)

In [ ]:
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])
t.numpy()

In [ ]:
np.array(t)

In [ ]:
tf.square(a)

In [ ]:
np.square(t)

### 타입 변환

In [ ]:
#tf.constant(2.0) + tf.constant(40)
try:
    tf.constant(2.0) + tf.constant(40)
except tf.errors.InvalidArgumentError as ex:
    print(ex)

In [ ]:
#tf.constant(2.0) + tf.constant(40., dtype=tf.float64)
try:
    tf.constant(2.0) + tf.constant(40., dtype=tf.float64)
except tf.errors.InvalidArgumentError as ex:
    print(ex)

#### 텐서는 타입의 자동변환이 없다.
* 성능저하 방지
* 사용자의 의지에 반하는 변환 방지

필요한경우 명시적으로 t.cast()를 이용하여 진행한다.

In [ ]:
t2 = tf.constant(40., dtype=tf.float64)

In [ ]:
tf.constant(2.0) + t2

#tf.constant(2.0) + tf.cast(t2, tf.float32)

### 문자열

tf.string은 문자열 텐서를 만들면 자동으로 UTF-8으로 인코딩하여 바이트문자열로 보관한다.

In [ ]:
s1=tf.constant("hello world")
s1

In [ ]:
s2=tf.constant("café")
s2

In [ ]:
u = tf.constant([ord(c) for c in "café"])
u

참고 :

`ord()` : Return the integer that represents the character


ord("h") => 주어진 문자 'h'에 대한 정수값을 반환한다.

출처: https://www.w3schools.com/python/ref_func_ord.asp

```정수 값과 UTF-8/ASCII 변환은 다음과 같이 할수 있다. ```

https://onlineutf8tools.com/convert-decimal-to-utf8


In [ ]:
b = tf.strings.unicode_encode(u, "UTF-8")
tf.strings.length(b, unit="UTF8_CHAR")

In [ ]:
tf.strings.unicode_decode(b, "UTF-8")

decode 하여 코드포인트를 구한다.

In [ ]:
tf.strings.unicode_encode(u, "UTF-8")

encode 하여 serialization에 사용하는 바이트열을 구한다.

### 문자열 배열

In [ ]:
p = tf.constant(["Café", "Coffee", "caffè", "咖啡"])

In [ ]:
tf.strings.length(p, unit="UTF8_CHAR")

In [ ]:
r = tf.strings.unicode_decode(p, "UTF8")
r

참고) Ragged tensors are the TensorFlow equivalent of nested variable-length lists

In [ ]:
print(r)

In [ ]:
print(r[3])

### 변수

지금까지 본 tf.Tensor는 변경이 불가능한 객체이다.

가중치중 파라미터는 학습중 변경이 되어야 하는데, 이를 위한 것이 변수, tf.Variable이 있다.

In [ ]:
v = tf.Variable([[1., 2., 3.], [4., 5., 6.]])
v

In [ ]:
v.assign(2 * v)
v

In [ ]:
v[0, 1].assign(42)

In [ ]:
v[:, 2].assign([0., 1.])

= 부호로 assign을 할 수 없다.

In [ ]:
v[1] = [7., 8., 9.]


In [ ]:
v[1].assign([7., 8., 9.])

In [ ]:
v.scatter_nd_update(indices=[[0, 0], [1, 2]],
                    updates=[100., 200.])

## Ragged Tensor(비정형) & Sparse Tensor(희소)


![Ragged Tensor vs Sparse](https://drive.google.com/uc?export=view&id=1QLNW_3ihPPMVviN-XVTahqPBERw_s5J4)Ragged Tensor vs Sparse

머신러닝을 위해 feature가 많은 즉 고차원의 vector로 되어 있는 데이터를 사용할 때, 벡터의 길이가 다르거나 벡터의 모든 차원중 많은 차원이 채워지지 않은 즉 0인 경우가 많다. 0을 포함한 모든 데이터를 메모리에 보관 하면, 필요 없이 많은 메모리 공간을 차지 하게 된다.

### 래그드 텐서
메모리를 절약하기 위해 의미 없는 데이터를 없앤 리스트의 리스트 형태로 표기하는 방식
* value_rowids
* row_lengths
* row_splits

처럼 3가지 방식으로 표기 할 수 있다.


value_rowids

In [ ]:
r1 = tf.RaggedTensor.from_value_rowids(
    values=[3, 1, 4, 1, 5, 9, 2, 6],
    value_rowids=[0, 0, 0, 0, 2, 2, 2, 3])
r1

row_lengths

In [ ]:
r2 = tf.RaggedTensor.from_row_lengths(
    values=[3, 1, 4, 1, 5, 9, 2, 6],
    row_lengths=[4, 0, 3, 1])
r2

row_splits

In [ ]:
r3 = tf.RaggedTensor.from_row_splits(
    values=[3, 1, 4, 1, 5, 9, 2, 6],
    row_splits=[0, 4, 4, 7, 8])
r3

row_splits = [ 0, 4, 4, 7, 8] 이 요소를 다음과 같이 해석한다.
* 0:4
* 4:4
* 4:7
* 7:8

In [ ]:
r3.to_tensor()

In [ ]:
r2.to_tensor()

예를 들어 앞의 스트링 예제에 나온 데이터를 살펴 보자.

In [ ]:
p = tf.constant(["Café", "Coffee", "caffè", "咖啡"])
r = tf.strings.unicode_decode(p, "UTF8")

In [ ]:
print(r[1])

In [ ]:
print(r)

In [ ]:
r2 = tf.ragged.constant([[65, 66], [], [67]])
print(tf.concat([r, r2], axis=0))

In [ ]:
r3 = tf.ragged.constant([[68, 69, 70], [71], [], [72, 73]])
print(tf.concat([r, r3], axis=1))

In [ ]:
tf.strings.unicode_encode(r3, "UTF-8")

In [ ]:
r.to_tensor()

### 희소 텐서
대부분이 0으로 채워진 텐서

In [ ]:
s = tf.SparseTensor(indices=[[0, 1], [1, 0], [2, 3]],
                    values=[1., 2., 3.],
                    dense_shape=[3, 4])

In [ ]:
print(s)

In [ ]:
tf.sparse.to_dense(s)

In [ ]:
s2 = s * 2.0
print('sparce : ', s2)

In [ ]:
tf.sparse.to_dense(s2)

In [ ]:
try:
    s3 = s + 1.
except TypeError as ex:
    print(ex)

SparseSensor에 스칼라 곱셈은 되지만, 스칼라 덧셈은 안된다. 다음과 같이 Dense로 바꾼후 덧셈을 할 수 있다.

In [ ]:
try:
    s3 = tf.sparse.to_dense(s) + 1.
except TypeError as ex:
    print(ex)
s3

tf.sparse.sparse_dens_matmul 은 벡터의 dot 프로덕트(내적) 계산에 해당한다.

In [ ]:
s4 = tf.constant([[10., 20.], [30., 40.], [50., 60.], [70., 80.]])
tf.sparse.sparse_dense_matmul(s, s4)

In [ ]:
print(tf.sparse.to_dense(s))
print(s4)

In [ ]:
s5 = tf.SparseTensor(indices=[[0, 2], [0, 1]],
                     values=[1., 2.],
                     dense_shape=[3, 4])
print(s5)

In [ ]:
try:
    s6 = tf.sparse.to_dense(s5)
except tf.errors.InvalidArgumentError as ex:
    print(ex)

s6

??? 왜 예외가 발생 했을까?

다음과 같이 살펴 보자

In [ ]:
s5 = tf.SparseTensor(indices=[[0, 1], [0, 2]],
                     values=[2., 1.],
                     dense_shape=[3, 4])
print(s5)

앞에 s5를 정의할때, indices가 [0,2], [0,1] 으로, 순차적이지 않아서 에러가 난것이다.

sparce tensor를 만들때, 인덱스는 순서가 정렬이 되어야 한다.

In [ ]:
s6 = tf.sparse.reorder(s5)
tf.sparse.to_dense(s6)

## 사용자 정의 손실 함수

지금까지 필기숫자 인식이나 패션이미지 인식등 뉴럴네트웍을 만들어 학습을 진행하면서, 다양한 손실(loss)함수를 사용하였다.

필기숫자 인식
<pre>
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
</pre>

패션이미지 인식
<pre>
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])
</pre>

그런데, **이러한 손실함수가 어떤 학습에 적합하지 않을때, 사용자가 손실함수를 만들어 사용할수 있다.** 여기서 Huber 손실함수를 만들어 사용해 보자.

캘리포니아 주택 데이터셋을 로드하여 준비해 보자. 먼저 이 데이터셋을 로드한 다음 훈련 세트, 검증 세트, 테스트 세트로 나눈다. 마지막으로 스케일을 변경한다:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
        housing.data, housing.target.reshape(-1, 1), random_state=42)

X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss  = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

In [ ]:
plt.figure(figsize=(8, 3.5))
z = np.linspace(-4, 4, 200)
plt.plot(z, huber_fn(0, z), "b-", linewidth=2, label="huber($z$)")
plt.plot(z, z**2 / 2, "b:", linewidth=1, label=r"$\frac{1}{2}z^2$")
plt.plot([-1, -1], [0, huber_fn(0., -1.)], "r--")
plt.plot([1, 1], [0, huber_fn(0., 1.)], "r--")
plt.gca().axhline(y=0, color='k')
plt.gca().axvline(x=0, color='k')
plt.axis([-4, 4, 0, 4])
plt.grid(True)
plt.xlabel("$z$")
plt.legend(fontsize=14)
plt.title("Huber loss", fontsize=14)
plt.show()

* b- => blue solid line
* b: => blue dotted line
* r-- => red dash line

In [ ]:
input_shape = X_train.shape[1:]

model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(30, activation="selu", kernel_initializer="lecun_normal"),
    keras.layers.Dense(1),
])

In [ ]:
model.compile(loss=huber_fn, optimizer="nadam", metrics=["mae"])

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

### 사용자 정의 요소를 가진 모델을 저장하고 로드하기



In [ ]:
model.save("my_model_with_a_custom_loss.keras")

**저장된 학습가중치를 로드 할때, keras가 기본으로 제공하지 않는 사용자 정의 손실함수인 경우, 함수를 맵핑할 필요가 있다.**

In [ ]:
model = keras.models.load_model("my_model_with_a_custom_loss.keras",
                                custom_objects={"huber_fn": huber_fn})

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

이전에 정의한 huber 함수는 -1 에서 1 사이의 오차를 작은 영역으로 정의하였다.

이 값(threshold)을 바꾸어야 할 경우에는 어떻게 할수 있을까?

In [ ]:
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < threshold
        squared_loss = tf.square(error) / 2
        linear_loss  = threshold * tf.abs(error) - threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    return huber_fn
# Python에서는 Variable이 데이터만 담을수 있는게 아니라 펑션을 가질수도 있다

In [ ]:
model.compile(loss=create_huber(2.0), optimizer="nadam", metrics=["mae"])

loss가 앞에서는 huber_fn 함수를 가르키고 있었지만, 지금은 create_huber(2.0) 함수가 리턴해주는 무엇인가를 가르키고 있다.

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

In [ ]:
model.save("my_model_with_a_custom_loss_threshold_2.keras")

그런데, 이 방식으로 하면, threshold값이 파이썬 코드에만 있어서 저장이 되질 않는다. 그래서 학습시 값을 알아와서 로드할때 전달하여야 한다.

그리고, 저장한 학습가중치를 load할때, 다음과 같이 앞에서 본 call을 하면, 이 threshold값을 전달 할수 없다.
<pre>
model = keras.models.load_model("my_model_with_a_custom_loss.keras",
                                custom_objects={"huber_fn": huber_fn})
</pre>

그래서 아래와 같이 학습시 값을 알아와서 아래와 같이 수행한다

In [ ]:
model = keras.models.load_model("my_model_with_a_custom_loss_threshold_2.keras",
                                custom_objects={"huber_fn": create_huber(2.0)})

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

#### 손실함수 Class

save할때와 load할때 파라미터 값을 주어야 하는데, 이게 save시와 load시에 다르게 주면 결과가 달라지게 된다.

학습을 계속해 나아갈수는 있으나, 이렇게 프로그램과 학습데이터 사이에 주요 변수가 분산이 되어 불편하다. 이를 keras.losses.Loss를 상속하여 만들어 해결할수 있다

In [ ]:
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)
    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss  = self.threshold * tf.abs(error) - self.threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(30, activation="selu", kernel_initializer="lecun_normal"),
    keras.layers.Dense(1),
])

In [ ]:
model.compile(loss=HuberLoss(3.), optimizer="nadam", metrics=["mae"])

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

In [ ]:
model.save("my_model_with_a_custom_loss_class.keras")

In [ ]:
m2 = keras.models.load_model("my_model_with_a_custom_loss_class.keras",
                                custom_objects={"HuberLoss": HuberLoss})

In [ ]:
m2.loss.threshold

load시에 별도로 파라미터를 주지 않았지만, save시의 파라미터가 세팅 되어 있는것을 알수 있다.

In [ ]:
m2.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

## 그외 사용자 정의 함수

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)
from tensorflow.keras.utils import register_keras_serializable

In [ ]:
# custom activation
def my_softplus(z):
    return tf.math.log(1.0 + tf.exp(z))

In [ ]:
# custom initializer
@register_keras_serializable()
class MyGlorotInitializer(keras.Initializer):
    def __call__(self, shape, dtype=tf.float32):
        stddev = tf.sqrt(2. / (shape[0] + shape[1]))
        return tf.random.normal(shape, stddev=stddev, dtype=dtype)
    def get_config(self):
        return {}
my_glorot_initializer = MyGlorotInitializer()


In [ ]:
# custom regularizer
@register_keras_serializable()
class MyL1Regularizer(keras.Regularizer):
    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(0.01 * weights))
    def get_config(self):
        return {}
my_l1_regularizer = MyL1Regularizer()


In [ ]:
# custom constraints
@register_keras_serializable()
class MyPositiveWeights(keras.constraints.Constraint):
    def __call__(self, weights):
        return tf.where(weights < 0., tf.zeros_like(weights), weights)
    def get_config(self):
        return {}
my_positive_weights = MyPositiveWeights()


custom function 을 가지고 layer를 만들어 보자

In [ ]:
layer = keras.layers.Dense(1, activation=my_softplus,
                           kernel_initializer=my_glorot_initializer,
                           kernel_regularizer=my_l1_regularizer,
                           kernel_constraint=my_positive_weights)

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(30, activation="selu", kernel_initializer="lecun_normal"),
    keras.layers.Dense(1, activation=my_softplus,
                       kernel_regularizer=my_l1_regularizer,
                       kernel_constraint=my_positive_weights,
                       kernel_initializer=my_glorot_initializer)
])

In [ ]:
model.compile(loss="mse", optimizer="nadam", metrics=["mae"])

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

In [ ]:
model.save("my_model_with_many_custom_parts.keras")

In [ ]:
model = keras.models.load_model(
    "my_model_with_many_custom_parts.keras",
    custom_objects={
       "my_l1_regularizer": my_l1_regularizer,
       "my_positive_weights": my_positive_weights,
       "my_glorot_initializer": my_glorot_initializer,
       "my_softplus": my_softplus,
    })

앞에 손실함수를 정의하는 방식과 같이 아래의 클래스들로 부터 상속을 받아 새로운 사용자 정의 함수를 만들어 사용 할수 있다.
* keras.regularizers.Regularizer
* keras.constraints.Constraint
* keras.initializers.Initializer

In [ ]:
class MyL1Regularizer(keras.regularizers.Regularizer):
    def __init__(self, factor):
        self.factor = factor
    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(self.factor * weights))
    def get_config(self):
        return {"factor": self.factor}

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(30, activation="selu", kernel_initializer="lecun_normal"),
    keras.layers.Dense(1, activation=my_softplus,
                       kernel_regularizer=MyL1Regularizer(0.01),
                       kernel_constraint=my_positive_weights,
                       kernel_initializer=my_glorot_initializer),
])

In [ ]:
model.compile(loss="mse", optimizer="nadam", metrics=["mae"])

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

In [ ]:
model.save("my_model_with_many_custom_parts.keras")

In [ ]:
model = keras.models.load_model(
    "my_model_with_many_custom_parts.keras",
    custom_objects={
       "MyL1Regularizer": MyL1Regularizer,
       "my_positive_weights": my_positive_weights,
       "my_glorot_initializer": my_glorot_initializer,
       "my_softplus": my_softplus,
    })

## 사용자 정의 지표

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(30, activation="selu", kernel_initializer="lecun_normal"),
    keras.layers.Dense(1),
])

In [ ]:
model.compile(loss="mse", optimizer="nadam", metrics=[create_huber(2.0)])

In [ ]:
model.fit(X_train_scaled, y_train, epochs=2)

**손실과 지표**

* 손실: 사람이 쉽게 이해를 못해도 경사하강법에서 사용하므로 미분가능, 그레디언트가 모든곳에서 0이 아니어야 함.
* 지표: 사람이 쉽게 이해하는게 중요하며, 미분이 안돼도 되고, 그레디언트가 0이 되어도 괜찮다.

다른 말로
* loss: 손실함수. 훈련셋과 연관. 훈련에 사용.
* metric: 평가지표. 검증셋과 연관. 훈련 과정을 모니터링하는데 사용.



**노트**: 손실과 지표에 같은 함수를 사용하면 다른 결과가 나올 수 있다. 이는 일반적으로 부동 소수점 정밀도 오차 때문이다다. 수학 식이 동일하더라도 연산은 동일한 순서대로 실행되지 않는다. 이로 인해 작은 차이가 발생한다. 또한 샘플 가중치를 사용하면 정밀도보다 더 큰 오차가 생긴다:

* 에포크에서 손실은 지금까지 본 모든 배치 손실의 평균이다. 각 배치 손실은 가중치가 적용된 샘플 손실의 합을 _배치 크기_ 로 나눈 것이다(샘플 가중치의 합으로 나눈 것이 아니다. 따라서 배치 손실은 손실의 가중 평균이 아니다).
* 에포크에서 지표는 가중치가 적용된 샘플 손실의 합을 지금까지 본 모든 샘플 가중치의 합으로 나눈 것이다. 다른 말로하면 모든 샘플 손실의 가중 평균이다. 따라서 위와 같지 않다.

수학적으로 말하면 손실 = 지표 * 샘플 가중치의 평균(더하기 약간의 부동 소수점 정밀도 오차)이다.

In [ ]:
model.compile(loss=create_huber(2.0), optimizer="nadam", metrics=[create_huber(2.0)])

In [ ]:
sample_weight = np.random.rand(len(y_train))
history = model.fit(X_train_scaled, y_train, epochs=2, sample_weight=sample_weight)

In [ ]:
history.history["loss"][0], history.history["huber_fn"][0] * sample_weight.mean()

### 스트리밍 지표
```
첫 배치 결과에서 5개를 True라 했는데 그중 4개를 맞추어 정밀도가 80%이고,
두번째 배치에서 3개를 True라 했는데, 모두 틀리면 정밀도가 0이 되어, 평균을 내면 40%가 된다.

그러나 두개 배치를 합쳐셔 보면, 8개의 True 예측치중 4개를 맞추어 50%의 정밀도를 보여준다.

이렇게 전체에 대한 정밀도를 계산하는 함수가 필요하고 이에 따라 나오는 지표를 스트리밍 지표라 한다.
```

복습:

<br><center><b>Confusion Matrix</b>
<img src='https://drive.google.com/uc?id=1uk5HdbJDgk22i5rSwQWljWkRCol0fGKu' height=400/></center>

---

**정밀도(Precision)** 는 참(Positive)으로 예측한 대상(FP + TP)중에 실제 참인값(TP)의 비율을 이야기 한다.
$$
\begin{align*}
정밀도(Precision) =
{\operatorname{TP}\over\operatorname{FP + TP}}
\end{align*}
$$
---


In [ ]:
precision = keras.metrics.Precision()
precision([0, 1, 1, 1, 0, 1, 0, 1], [1, 1, 0, 1, 0, 1, 0, 1])

In [ ]:
precision([0, 1, 0, 0, 1, 0, 1, 1], [1, 0, 1, 1, 0, 0, 0, 0])

In [ ]:
precision.result()

In [ ]:
precision.variables

<center><b>Precision = True Positives / (True Positives + False Positives)</b></center>

In [ ]:
precision.reset_state()

In [ ]:
precision([1,0,1 ], [1,0,1])

In [ ]:
precision.result()

In [ ]:
precision.variables

스트리밍 지표 만들기:

In [ ]:
class HuberMetric(keras.metrics.Metric):
    def __init__(self, threshold=1.0, **kwargs):
        super().__init__(**kwargs) # 기본 매개변수 처리 (예를 들면, dtype)
        self.threshold = threshold
        self.huber_fn = create_huber(threshold)
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")
    def update_state(self, y_true, y_pred, sample_weight=None):
        metric = self.huber_fn(y_true, y_pred)
        self.total.assign_add(tf.reduce_sum(metric))
        self.count.assign_add(tf.cast(tf.size(y_true), tf.float32))
    def result(self):
        return self.total / self.count
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

In [ ]:
m = HuberMetric(2.)

# total = 2 * |10 - 2| - 2²/2 = 14
# count = 1
# result = 14 / 1 = 14
m(tf.constant([[2.]]), tf.constant([[10.]]))

In [ ]:
# total = total + (|1 - 0|² / 2) + (2 * |9.25 - 5| - 2² / 2) = 14 + 7 = 21
# count = count + 2 = 3
# result = total / count = 21 / 3 = 7
m(tf.constant([[0.], [5.]]), tf.constant([[1.], [9.25]]))

m.result()

In [ ]:
m.variables

In [ ]:
m.reset_state()
m.variables

In [ ]:
m.reset_state()
m(tf.constant([[0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.]]), tf.constant([[5.], [5.], [5.], [5.], [5.], [5.], [5.], [5.]]))

In [ ]:
m.result()

In [ ]:
m.variables

In [ ]:
m.reset_state()
m(tf.constant([[0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.]]), tf.constant([[6.], [6.], [6.], [6.], [6.], [6.], [6.], [6.]]))

\`HuberMetric` 클래스가 잘 동작하는지 확인해 보자:

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(30, activation="selu", kernel_initializer="lecun_normal"),
    keras.layers.Dense(1),
])

In [ ]:
model.compile(loss=create_huber(2.0), optimizer="nadam", metrics=[HuberMetric(2.0)])

In [ ]:
model.fit(X_train_scaled.astype(np.float32), y_train.astype(np.float32), epochs=2)

In [ ]:
model.save("my_model_with_a_custom_metric.keras")

In [ ]:
model = keras.models.load_model("my_model_with_a_custom_metric.keras",
                                custom_objects={"huber_fn": create_huber(2.0),
                                                "HuberMetric": HuberMetric})

In [ ]:
model.fit(X_train_scaled.astype(np.float32), y_train.astype(np.float32), epochs=2)